<a href="https://colab.research.google.com/github/ajaykumar080286/PyTorch/blob/main/5_pytorch_training_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [3]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [4]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [5]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [6]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [7]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [11]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [12]:
X_train_tensor.shape

torch.Size([455, 30])

In [13]:
X_train.shape

(455, 30)

**Defining the model**

In [43]:
class MySimpleNN():

  def __init__(self,X):

    self.weights=torch.rand(X.shape[1],dtype=torch.float64,requires_grad=True)
    self.bias=torch.zeros(1,dtype=torch.float64, requires_grad=True)

    #forward pass

  def forward_pass(self,X):

    z=torch.matmul(X,self.weights)+self.bias
    y_pred=torch.sigmoid(z)
    return y_pred

  def loss_function(self, y_pred, y):
    # Clamp predictions to avoid log(0)
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    # Calculate loss
    loss = -(y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor) * torch.log(1 - y_pred)).mean()
    return loss


In [70]:
learning_rate = 0.1
epochs = 25

**Training Pipeline**

In [71]:
model=MySimpleNN(X_train_tensor)

for epoch in range(epochs):
  y_pred=model.forward_pass(X_train_tensor)

  # loss
  loss = model.loss_function(y_pred, y_train_tensor)

  # backward pass
  loss.backward()

   # parameters update
  with torch.no_grad():
    model.weights -= learning_rate * model.weights.grad
    model.bias -= learning_rate * model.bias.grad

  # zero gradients
  model.weights.grad.zero_()
  model.bias.grad.zero_()

# print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.6945174231647657
Epoch: 2, Loss: 0.6754163278306787
Epoch: 3, Loss: 0.6553895076129328
Epoch: 4, Loss: 0.635986629293
Epoch: 5, Loss: 0.6171846380876175
Epoch: 6, Loss: 0.5989622255235353
Epoch: 7, Loss: 0.581299458656925
Epoch: 8, Loss: 0.5641774454672442
Epoch: 9, Loss: 0.5475781011174538
Epoch: 10, Loss: 0.5314840557937648
Epoch: 11, Loss: 0.5158787165908624
Epoch: 12, Loss: 0.5007464718100192
Epoch: 13, Loss: 0.4860730053619476
Epoch: 14, Loss: 0.47184567355803486
Epoch: 15, Loss: 0.45805388005467756
Epoch: 16, Loss: 0.4446893686109013
Epoch: 17, Loss: 0.43174633273424834
Epoch: 18, Loss: 0.41922122772921777
Epoch: 19, Loss: 0.40711218322415393
Epoch: 20, Loss: 0.3954179798289385
Epoch: 21, Loss: 0.3841366878347965
Epoch: 22, Loss: 0.3732642368295892
Epoch: 23, Loss: 0.36279330202401844
Epoch: 24, Loss: 0.35271285273270375
Epoch: 25, Loss: 0.3430084907593071


In [72]:
model.weights

tensor([ 0.4440,  0.5220,  0.1850,  1.1475,  0.2792,  0.3531,  0.4172,  0.0550,
         0.6843,  0.5465,  0.5804, -0.0516,  0.0435,  0.2551,  0.5347, -0.1614,
         0.6099,  0.2661,  0.6771,  0.1157,  0.9672,  0.3809,  0.3238,  0.7514,
         0.8077,  0.5659,  0.7065,  0.8376,  0.5169,  0.7525],
       dtype=torch.float64, requires_grad=True)

In [73]:
model.bias

tensor([-0.0874], dtype=torch.float64, requires_grad=True)

**Evaluation**

In [88]:
with torch.no_grad():
  y_pred=model.forward_pass(X_test_tensor)
  y_pred=(y_pred>0.9).float()
  accuracy=(y_pred==y_test_tensor).float().mean()
  print(f'accuracy:{accuracy.item()}')

accuracy:0.9035087823867798
